In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
orders = pd.read_csv("../data/processed/orders_cleaned.csv")
customers = pd.read_csv("../data/processed/customers_cleaned.csv")
order_items = pd.read_csv("../data/processed/order_items_cleaned.csv")
products = pd.read_csv("../data/processed/products_cleaned.csv")
sellers = pd.read_csv("../data/processed/sellers_cleaned.csv")
payments = pd.read_csv("../data/processed/payments_cleaned.csv")
reviews = pd.read_csv("../data/processed/reviews_cleaned.csv")
geolocation = pd.read_csv("../data/processed/geolocation_cleaned.csv")
categories = pd.read_csv("../data/processed/category_translation_cleaned.csv")

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [3]:
datasets = {
    "Orders": orders,
    "Customers": customers,
    "Order Items": order_items,
    "Products": products,
    "Sellers": sellers,
    "Payments": payments,
    "Reviews": reviews,
    "Geolocation": geolocation,
    "Categories": categories
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Orders: (99441, 11)
Customers: (99441, 5)
Order Items: (112650, 8)
Products: (32951, 9)
Sellers: (3095, 4)
Payments: (103886, 5)
Reviews: (99224, 8)
Geolocation: (738332, 5)
Categories: (71, 2)


In [4]:
product_table = products.merge(
    categories,
    on="product_category_name",
    how="left"
)

print("Product table shape:", product_table.shape)

display(product_table.head())

Product table shape: (32951, 10)


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


In [5]:
product_table.to_csv(
    "../data/processed/product_table.csv",
    index=False
)

In [6]:
sales_table = order_items.merge(
    orders,
    on="order_id",
    how="left"
)

sales_table = sales_table.merge(
    product_table,
    on="product_id",
    how="left"
)

sales_table = sales_table.merge(
    sellers,
    on="seller_id",
    how="left",
    suffixes=("", "_seller")
)

print("Sales table shape:", sales_table.shape)

display(sales_table.head())

Sales table shape: (112650, 30)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,total_item_value,customer_id,order_status,...,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,72.19,3ce436f183e68e07877b285a838db11a,delivered,...,598.0,4.0,650.0,28.0,9.0,14.0,cool_stuff,27277,VOLTA REDONDA,SP
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,259.83,f6dd3ec061db4e3987629fe6b26e5cce,delivered,...,239.0,2.0,30000.0,50.0,30.0,40.0,pet_shop,3471,SAO PAULO,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,216.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,...,695.0,2.0,3050.0,33.0,13.0,33.0,furniture_decor,37564,BORDA DA MATA,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,25.78,d4eb9395c8c0431ee92fce09860c5a06,delivered,...,480.0,1.0,200.0,16.0,10.0,15.0,perfumery,14403,FRANCA,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,218.04,58dbd0b2d70206bf40e62cd34e84d795,delivered,...,409.0,1.0,3750.0,35.0,40.0,30.0,garden_tools,87900,LOANDA,PR


In [7]:
sales_table.to_csv(
    "../data/processed/sales_table.csv",
    index=False
)

In [8]:
customer_table = customers.merge(
    orders,
    on="customer_id",
    how="left"
)

print("Customer table shape:", customer_table.shape)

display(customer_table.head())

Customer table shape: (99441, 15)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,delivery_status
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,FRANCA,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,2017-05-25 10:35:35,2017-06-05,8.0,-11.0,On Time
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,SAO BERNARDO DO CAMPO,SP,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,2018-01-29 12:41:19,2018-02-06,16.0,-8.0,On Time
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,SAO PAULO,SP,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,2018-06-14 17:58:51,2018-06-13,26.0,1.0,Delayed
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,MOGI DAS CRUZES,SP,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,2018-03-28 16:04:25,2018-04-10,14.0,-13.0,On Time
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,CAMPINAS,SP,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,2018-08-09 20:55:48,2018-08-15,11.0,-6.0,On Time


In [9]:
customer_table.to_csv(
    "../data/processed/customer_table.csv",
    index=False
)

In [10]:
delivery_table = orders.merge(
    customers,
    on="customer_id",
    how="left"
)

delivery_table = delivery_table.merge(
    order_items[
        [
            "order_id",
            "seller_id",
            "product_id",
            "price",
            "freight_value"
        ]
    ],
    on="order_id",
    how="left"
)

delivery_table = delivery_table.merge(
    sellers,
    on="seller_id",
    how="left",
    suffixes=("", "_seller")
)

print("Delivery table shape:", delivery_table.shape)

display(delivery_table.head())

Delivery table shape: (113425, 22)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,...,customer_zip_code_prefix,customer_city,customer_state,seller_id,product_id,price,freight_value,seller_zip_code_prefix,seller_city,seller_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,...,3149,SAO PAULO,SP,3504c0cb71d7fa48d967e0e4c94d59d9,87285b34884572647811a353c7ac498a,29.99,8.72,9350.0,MAUA,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,-6.0,...,47813,BARREIRAS,BA,289cdb325fb7e7f891c38608bf9e0962,595fac2a385ac33a80bd5114aec74eb8,118.70,22.76,31570.0,BELO HORIZONTE,SP
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,-18.0,...,75265,VIANOPOLIS,GO,4869f7a5dfa277a7dca6462dcf3b52b2,aa4383b373c6aca5d8797843e5594415,159.90,19.22,14840.0,GUARIBA,SP
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0,-13.0,...,59296,SAO GONCALO DO AMARANTE,RN,66922902710d126a0e7d26b0e3805106,d0b61bfb1de832b15ba9d266ca96e5b0,45.00,27.20,31842.0,BELO HORIZONTE,MG
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0,-10.0,...,9195,SANTO ANDRE,SP,2c9e548be18521d1c43cde1c582c6de8,65266b2da20d04dbe00c5c2d3bb7859e,19.90,8.72,8752.0,MOGI DAS CRUZES,SP


In [11]:
delivery_table.to_csv(
    "../data/processed/delivery_table.csv",
    index=False
)

In [12]:
review_table = orders.merge(
    reviews,
    on="order_id",
    how="left"
)

print("Review table shape:", review_table.shape)

display(review_table.head())

Review table shape: (99992, 18)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,delivery_status,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,review_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,On Time,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,Positive
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,-6.0,On Time,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50,Positive
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,-18.0,On Time,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58,Positive
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0,-13.0,On Time,359d03e676b3c069f62cadba8dd3f6e8,5.0,NaN,O produto foi exatamente o que eu esperava e e...,2017-12-03 00:00:00,2017-12-05 19:21:58,Positive
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0,-10.0,On Time,e50934924e227544ba8246aeb3770dd4,5.0,NaN,NaN,2018-02-17 00:00:00,2018-02-18 13:02:51,Positive


In [13]:
review_table.to_csv(
    "../data/processed/review_table.csv",
    index=False
)

In [14]:
payment_table = orders.merge(
    payments,
    on="order_id",
    how="left"
)

print("Payment table shape:", payment_table.shape)

display(payment_table.head())

Payment table shape: (103887, 15)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,delivery_status,payment_sequential,payment_type,payment_installments,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,On Time,1.0,credit_card,1.0,18.12
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,On Time,3.0,voucher,1.0,2.00
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,On Time,2.0,voucher,1.0,18.59
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,-6.0,On Time,1.0,boleto,1.0,141.46
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,-18.0,On Time,1.0,credit_card,3.0,179.12


In [15]:
payment_table.to_csv(
    "../data/processed/payment_table.csv",
    index=False
)

In [16]:
print("Sales table:")
print(sales_table.shape)

print("\nCustomer table:")
print(customer_table.shape)

print("\nDelivery table:")
print(delivery_table.shape)

print("\nReview table:")
print(review_table.shape)

print("\nPayment table:")
print(payment_table.shape)

Sales table:
(112650, 30)

Customer table:
(99441, 15)

Delivery table:
(113425, 22)

Review table:
(99992, 18)

Payment table:
(103887, 15)


In [17]:
print("Missing order IDs in sales:")
print(sales_table["order_id"].isna().sum())

print("\nMissing product IDs in sales:")
print(sales_table["product_id"].isna().sum())

print("\nMissing seller IDs in sales:")
print(sales_table["seller_id"].isna().sum())

Missing order IDs in sales:
0

Missing product IDs in sales:
0

Missing seller IDs in sales:
0


In [18]:
output_files = [
    "product_table.csv",
    "sales_table.csv",
    "customer_table.csv",
    "delivery_table.csv",
    "review_table.csv",
    "payment_table.csv"
]

for file in output_files:
    path = "../data/processed/" + file
    
    if os.path.exists(path):
        print("Created:", file)
    else:
        print("Missing:", file)

Created: product_table.csv
Created: sales_table.csv
Created: customer_table.csv
Created: delivery_table.csv
Created: review_table.csv
Created: payment_table.csv
